# Water Quality Prediction: XGBoost Regression (Improved)

This Snowflake notebook keeps the benchmark data workflow but improves the XGBoost training loop for better generalization.

**Key improvements included:**

- Uses **pandas DataFrames end-to-end** (preserves feature names/order; avoids common XGBoost feature mismatch errors).

- Removes **StandardScaler** (tree boosting does not require scaling).

- Adds **Train/Validation/Test split** and **Early Stopping** (via XGBoost callbacks for version compatibility).

- Fits **median imputer on training data only** (reduces leakage).

- Optional **log1p transform** for skewed targets (enabled for DRP by default).

**Targets:** Total Alkalinity (TA), Electrical Conductance (EC), Dissolved Reactive Phosphorus (DRP)

**Reminder:** Do not use Latitude/Longitude directly as model features per challenge guidance.

## 1) Install dependencies (Snowflake container runtime)
Run once per fresh container session. Restart kernel if required.

In [ ]:
# If running on Snowflake container runtime, install packages from requirements.txt
# (If requirements.txt is already present in the left Files panel, this will work as-is.)
!pip install uv
!uv pip install -r "../requirements.txt"

# Ensure xgboost is available
!uv pip install -U xgboost

## 2) Imports

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error

try:
    from IPython.display import display
except Exception:
    display = print

from xgboost import XGBRegressor
try:
    from xgboost.callback import EarlyStopping
except Exception:
    EarlyStopping = None

## 3) Load training data
This expects the same CSVs used by the benchmark notebook to be in the notebook Files panel:
- `water_quality_training_dataset.csv`
- `landsat_features_training.csv`
- `terraclimate_features_training.csv`

In [ ]:
Water_Quality_df = pd.read_csv('../water_quality_training_dataset.csv')
landsat_train_features = pd.read_csv('../landsat_features_training_full.csv')
Terraclimate_df = pd.read_csv('../terraclimate_features_training.csv')

# Ensure NDMI/MNDWI are numeric if present
for col in ['NDMI', 'MNDWI']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = pd.to_numeric(landsat_train_features[col], errors='coerce')

display(Water_Quality_df.head())
display(landsat_train_features.head())
display(Terraclimate_df.head())

## 4) Join datasets
We concatenate columns (dropping duplicate column names).

In [ ]:
def combine_three_datasets(dataset1, dataset2, dataset3):
    data = pd.concat([dataset1, dataset2, dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_three_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)
display(wq_data.head())

## 5) Select features + targets
By default we use the baseline set: `swir22`, `NDMI`, `MNDWI`, `pet`.

You can optionally add more columns (if present) by setting `USE_EXTENDED_FEATURES = True`.

In [ ]:
# Baseline feature set
BASE_FEATURE_COLS = ['swir22', 'NDMI', 'MNDWI', 'pet']

# Optional: enable and edit this list if your CSVs contain more useful features
USE_EXTENDED_FEATURES = True
EXTENDED_FEATURE_CANDIDATES = [
    # Landsat/Sentinel-style bands/indices (keep only those that exist in your data)
    'blue', 'green', 'red', 'nir', 'swir16',
    'NDVI', 'EVI', 'SAVI', 'NBR', 'NDWI',
    # TerraClimate candidates (examples; keep only those that exist)
    'pr', 'ppt', 'tmin', 'tmax', 'tmean', 'soil', 'srad', 'aet', 'def', 'vap'
]

TARGET_COLS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# Resolve features that actually exist, and explicitly forbid Lat/Lon as features
FORBIDDEN_FEATURES = set(['Latitude', 'Longitude', 'lat', 'lon', 'LATITUDE', 'LONGITUDE'])

def resolve_feature_cols(df, base_cols, extended=False, extended_candidates=None):
    cols = [c for c in base_cols if c in df.columns]
    if extended and extended_candidates:
        cols += [c for c in extended_candidates if c in df.columns]
    # remove forbidden
    cols = [c for c in cols if c not in FORBIDDEN_FEATURES]
    # stable unique order
    seen = set()
    out = []
    for c in cols:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

FEATURE_COLS = resolve_feature_cols(wq_data, BASE_FEATURE_COLS, USE_EXTENDED_FEATURES, EXTENDED_FEATURE_CANDIDATES)

print('Using feature columns:', FEATURE_COLS)
print('Targets:', TARGET_COLS)

# Build modeling frame, but keep full rows for imputation later
model_df = wq_data[FEATURE_COLS + TARGET_COLS].copy()
X_full = model_df[FEATURE_COLS]
Y = {
    'Total Alkalinity': model_df['Total Alkalinity'],
    'Electrical Conductance': model_df['Electrical Conductance'],
    'Dissolved Reactive Phosphorus': model_df['Dissolved Reactive Phosphorus']
}

display(model_df.head())

## 6) Training utilities (impute, split, early stopping, evaluation)
- Keep X as a DataFrame to preserve feature names/order.
- Fit median imputer on training split only.
- Use a validation split + early stopping via callbacks (works across many xgboost versions).

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def evaluate(y_true, y_pred, label):
    r2 = float(r2_score(y_true, y_pred))
    e = rmse(y_true, y_pred)
    print(f'{label}  R2={r2:.4f}  RMSE={e:.4f}')
    return r2, e

def make_model(seed=42):
    # A stronger generalization-friendly starter config
    return XGBRegressor(
        n_estimators=8000,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=10,
        subsample=0.75,
        colsample_bytree=0.75,
        reg_lambda=3.0,
        reg_alpha=0.2,
        gamma=0.5,
        objective='reg:squarederror',
        random_state=seed,
        tree_method='hist'
    )

def train_one_target(X_df, y_series, target_name, use_log1p=False, seed=42):
    print('' + '='*70)
    print('Training target:', target_name, '| log1p:', use_log1p)
    print('='*70)

    # Split: train/valid/test
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X_df, y_series, test_size=0.2, random_state=seed
    )
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=seed
    )

    # Impute (fit on train only)
    imputer = SimpleImputer(strategy='median')
    X_train_i = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_valid_i = pd.DataFrame(imputer.transform(X_valid), columns=X_valid.columns, index=X_valid.index)
    X_test_i  = pd.DataFrame(imputer.transform(X_test),  columns=X_test.columns,  index=X_test.index)

    model = make_model(seed=seed)

    # Optional log transform
    if use_log1p:
        y_train_fit = np.log1p(y_train)
        y_valid_fit = np.log1p(y_valid)
        y_test_true = y_test
    else:
        y_train_fit = y_train
        y_valid_fit = y_valid
        y_test_true = y_test

    fit_kwargs = dict(
        eval_set=[(X_valid_i, y_valid_fit)],
        verbose=False
    )

    # Prefer callbacks early stopping for compatibility
    if EarlyStopping is not None:
        fit_kwargs['callbacks'] = [EarlyStopping(rounds=200, save_best=True)]
    else:
        # Fallback: no early stopping available
        pass

    model.fit(X_train_i, y_train_fit, **fit_kwargs)

    # Predictions
    pred_train = model.predict(X_train_i)
    pred_test  = model.predict(X_test_i)

    if use_log1p:
        pred_train = np.expm1(pred_train)
        pred_test = np.expm1(pred_test)

    # Evaluate
    r2_tr, rmse_tr = evaluate(y_train, pred_train, 'Train')
    r2_te, rmse_te = evaluate(y_test_true, pred_test,  'Test ')

    metrics = {
        'Target': target_name,
        'UseLog1p': bool(use_log1p),
        'R2_Train': r2_tr,
        'RMSE_Train': rmse_tr,
        'R2_Test': r2_te,
        'RMSE_Test': rmse_te
    }

    # Refit a final model for submission using train_full with a validation split for early stopping
    # (This typically generalizes better than fitting on all data without early stopping.)
    X_tr, X_va, y_tr, y_va = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=seed)
    imputer_final = SimpleImputer(strategy='median')
    X_tr_i = pd.DataFrame(imputer_final.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_va_i = pd.DataFrame(imputer_final.transform(X_va), columns=X_va.columns, index=X_va.index)
    model_final = make_model(seed=seed)

    if use_log1p:
        y_tr_fit = np.log1p(y_tr)
        y_va_fit = np.log1p(y_va)
    else:
        y_tr_fit = y_tr
        y_va_fit = y_va

    fit_kwargs2 = dict(eval_set=[(X_va_i, y_va_fit)], verbose=False)
    if EarlyStopping is not None:
        fit_kwargs2['callbacks'] = [EarlyStopping(rounds=200, save_best=True)]
    model_final.fit(X_tr_i, y_tr_fit, **fit_kwargs2)

    return {
        'model': model_final,
        'imputer': imputer_final,
        'metrics': metrics
    }

## 7) Train + evaluate (3 targets)

In [ ]:
artifacts = {}

# Enable log1p where it commonly helps (DRP is usually skewed).
LOG1P_TARGETS = {
    'Total Alkalinity': False,
    'Electrical Conductance': False,
    'Dissolved Reactive Phosphorus': True
}

for tgt, y in Y.items():
    artifacts[tgt] = train_one_target(X_full, y, tgt, use_log1p=LOG1P_TARGETS.get(tgt, False), seed=42)

results_summary = pd.DataFrame([artifacts[t]['metrics'] for t in artifacts])
display(results_summary)
print('Mean R2 (Test) across 3 targets:', float(results_summary['R2_Test'].mean()))

## 8) Create `submission.csv` using validation features (optional)
Requires these files in the notebook Files panel:
- `submission_template.csv`
- `landsat_features_validation.csv`
- `terraclimate_features_validation.csv`

In [ ]:
test_file = pd.read_csv('../submission_template.csv')
landsat_val_features = pd.read_csv('../landsat_features_validation.csv')
terraclimate_val_df = pd.read_csv('../terraclimate_features_validation.csv')

for col in ['NDMI', 'MNDWI']:
    if col in landsat_val_features.columns:
        landsat_val_features[col] = pd.to_numeric(landsat_val_features[col], errors='coerce')

# Build validation feature frame using the same FEATURE_COLS
val_raw = combine_three_datasets(test_file, landsat_val_features, terraclimate_val_df)
X_val = val_raw[FEATURE_COLS].copy()

# Predict each target using its own imputer/model
preds = {}
for tgt in TARGET_COLS:
    art = artifacts[tgt]
    model = art['model']
    imp = art['imputer']
    X_val_i = pd.DataFrame(imp.transform(X_val), columns=X_val.columns)
    p = model.predict(X_val_i)
    if LOG1P_TARGETS.get(tgt, False):
        p = np.expm1(p)
    preds[tgt] = p

submission_df = pd.DataFrame({
    'Longitude': test_file['Longitude'].values,
    'Latitude': test_file['Latitude'].values,
    'Sample Date': test_file['Sample Date'].values,
    'Total Alkalinity': preds['Total Alkalinity'],
    'Electrical Conductance': preds['Electrical Conductance'],
    'Dissolved Reactive Phosphorus': preds['Dissolved Reactive Phosphorus']
})

display(submission_df.head())
submission_df.to_csv('/tmp/xgboost_submission_improved.csv', index=False)
print('Wrote /tmp/xgboost_submission_improved.csv')

### Optional: Upload submission file in Snowflake
Uncomment and run if you want to PUT the file into the notebook stage (same pattern as the benchmark notebook).

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
# session.sql(f"""
# PUT file:///tmp/xgboost_submission_improved.csv
# 'snow://workspace/USER$.PUBLIC.\"snowflake-challenge\"/versions/live/submission'
# AUTO_COMPRESS=FALSE
# OVERWRITE=TRUE
# """).collect()
# print('File saved! Refresh the browser to see it in the sidebar')